In [6]:
import os
print(os.listdir('.'))

['.config', 'country-list(2).csv', 'city_temperature(2).csv', 'sample_data']


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, BooleanType
import os

spark = SparkSession.builder.appName("HW3_Part1").getOrCreate()

# Helper to save results as a single text/csv file for your submission
def save_df(df, folder_name):
    if os.path.exists(folder_name):
        os.system(f"rm -rf {folder_name}")
    # Coalesce(1) ensures all data is in one part-00000 file
    df.coalesce(1).write.option("header", "false").option("sep", "\t").mode("overwrite").csv(folder_name)

# Load Data
temp_df = spark.read.csv("city_temperature(2).csv", header=True, inferSchema=True)
country_df = spark.read.csv("country-list(2).csv", header=True, inferSchema=True)

# Clean Data: Remove placeholder -99 values
temp_df = temp_df.filter(temp_df.AvgTemperature > -90)

In [11]:
# A. Avg Temperature per Region
res_a = temp_df.groupBy("Region").avg("AvgTemperature")
save_df(res_a, "q1_partA_results")

# B. Avg Temp by Year for "Africa"
res_b = temp_df.filter(temp_df.Region == "Africa") \
               .groupBy("Year").avg("AvgTemperature")
save_df(res_b, "q1_partB_results")

# C. Avg Temp by City in "Jordan"
res_c = temp_df.filter(temp_df.Country == "Jordan") \
               .groupBy("City").avg("AvgTemperature")
save_df(res_c, "q1_partC_results")

In [12]:
# --- Part D: Capital City Average (Fixed for Ambiguity) ---
joined_d = temp_df.join(country_df,
                        (temp_df.City == country_df.capital) &
                        (temp_df.Country == country_df.country)) \
                  .drop(country_df.country) # This removes the duplicate column

res_d = joined_d.groupBy("capital", "Country").avg("AvgTemperature") \
                .select("capital", "Country", "avg(AvgTemperature)")

save_df(res_d, "q1_partD_results")

# --- Part E: Broadcast Variable (Fixed for Ambiguity) ---
from pyspark.sql.functions import broadcast

res_e = temp_df.join(broadcast(country_df),
                     (temp_df.City == country_df.capital) &
                     (temp_df.Country == country_df.country)) \
               .drop(country_df.country) # Remove ambiguity here too

res_e = res_e.groupBy("capital", "Country").avg("AvgTemperature") \
             .select("capital", "Country", "avg(AvgTemperature)")

save_df(res_e, "q1_partE_results")

# --- Part F: UDFs (Fixed for Ambiguity) ---
# i. Filter years >= 2012
filter_year_udf = F.udf(lambda year: year >= 2012, BooleanType())
temp_f_filtered = temp_df.filter(filter_year_udf(F.col("Year")))

# Join and solve ambiguity
res_f_i_data = temp_f_filtered.join(country_df,
                                    (temp_f_filtered.City == country_df.capital) &
                                    (temp_f_filtered.Country == country_df.country)) \
                              .drop(country_df.country)

res_f_i = res_f_i_data.groupBy("capital", "Country").avg("AvgTemperature")
save_df(res_f_i, "q1_partF_i_results")

# ii. Format output string
def format_output(capital, country, avg_temp):
    return f"{capital} is the capital of {country} and its average temperature is {avg_temp:.2f}"

format_udf = F.udf(format_output, StringType())

res_f_ii = res_f_i.select(format_udf(F.col("capital"),
                                     F.col("Country"),
                                     F.col("avg(AvgTemperature)")).alias("final_output"))
save_df(res_f_ii, "q1_partF_ii_results")